# COMMUNITY PERSONA IDENTIFICATION — COLAB ENTERPRISE (SCHEDULED)

**Platform**: Multi-platform (Instagram, TikTok, Facebook)
**Runtime**: Google Colab Enterprise (headless, scheduled) + Vertex AI (Gemini)
**Version**: 3.0 — corrected & cloud-ready

This is the **scheduled / headless** sibling of `persona_fix.ipynb`. Differences:
- All outputs persist to **GCS** (Colab disk is ephemeral).
- `print` → `logging`; core run wrapped in a fatal-error guard that re-raises so
  the scheduler marks the run FAILED instead of a silent broken success.
- Implicit **Application Default Credentials** (service account) — no key files.
- The missing **Vertex init** and **data-loading** cells are restored; the
  duplicate aggregation cell is gone; the ML loader reads the **unified**
  `comments_ml.parquet` and filters by `platform`.

## Run model (human-in-the-loop, scheduler-friendly)
- `PIPELINE_MODE="SAMPLE"` → Stage 0 + Stage 1, writes taxonomy candidates to GCS, exits.
- *Human edits the GCS taxonomy.json → `final_taxonomy` + `status="APPROVED"`.*
- `PIPELINE_MODE="FULL"` → Stage 0 + Stage 2, classifies all users, writes parquet to GCS.

`PIPELINE_MODE` reads from an env var, so a Colab Enterprise schedule can set it
per job without editing the notebook.

### Cells
1. Install deps · 2. Config + logging + GCS helpers · 3. Vertex AI init
4. Load comments (LLM jsonl) · 5. Feature engineering (unified ML parquet)
6. User aggregation · 7. Stage 1 · 8. Stage 2 · 9. Orchestrator (guarded) · 10. Validation

In [ ]:
# Cell 1 — Dependencies (non-builtin only; gcsfs lets pandas read/write gs://)
!pip install -q google-cloud-aiplatform pandas numpy pyarrow tqdm emoji gcsfs

In [ ]:
# Cell 2 — Configuration, logging, GCS helpers
import os, re, json, time, sys, logging, traceback
from typing import List, Dict, Optional, Any

import pandas as pd
import numpy as np
from tqdm import tqdm
import emoji as emoji_lib

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("persona")

# ── GCP / Vertex (implicit ADC on Colab Enterprise — no key files) ────────────
GCP_PROJECT_ID = "gen-lang-client-0792749758"
GCP_LOCATION   = "us-central1"
GCS_BUCKET     = "afb_showreel"

# ── Models / determinism ──────────────────────────────────────────────────────
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"
TEMPERATURE = 0.0
TOP_P       = 1.0
MAX_OUTPUT_TOKENS_STAGE1 = 2048
MAX_OUTPUT_TOKENS_STAGE2 = 512

# ── Mode (scheduler can override via env var) ─────────────────────────────────
PIPELINE_MODE = os.environ.get("PIPELINE_MODE", "SAMPLE")  # SAMPLE | FULL | ALL

# ── Inputs (data-prep outputs: ONE unified file per format, filter by platform)─
BASE_PREFIX       = "comment_prep/2131052821112422400/final_output"
COMMENTS_LLM_PATH = f"gs://{GCS_BUCKET}/{BASE_PREFIX}/comments_llm.jsonl"
COMMENTS_ML_PATH  = f"gs://{GCS_BUCKET}/{BASE_PREFIX}/comments_ml.parquet"
PLATFORMS_TO_INCLUDE = ["instagram", "tiktok", "facebook"]

# ── Sampling / batching ───────────────────────────────────────────────────────
SAMPLE_N_USERS    = 500
SAMPLE_SEED       = 42
BATCH_SIZE_STAGE1 = 20
BATCH_SIZE_STAGE2 = 10

# ── Outputs (persist to GCS — Colab disk is ephemeral) ────────────────────────
OUTPUT_PREFIX       = "persona_pipeline/2131052821112422400"
TAXONOMY_GCS_PATH   = f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/taxonomy.json"
RESULTS_GCS_PATH    = f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/user_personas.parquet"
CHECKPOINT_GCS_PATH = f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/_stage2_checkpoint.parquet"

# ── Small GCS text/JSON helpers (parquet uses pandas+gcsfs directly) ──────────
from google.cloud import storage
_gcs_client = storage.Client(project=GCP_PROJECT_ID)

def _split_gs(path: str):
    assert path.startswith("gs://"), f"not a gs:// path: {path}"
    bucket, _, key = path[5:].partition("/")
    return bucket, key

def gcs_write_text(path: str, text: str):
    b, k = _split_gs(path)
    _gcs_client.bucket(b).blob(k).upload_from_string(text)

def gcs_read_text(path: str) -> str:
    b, k = _split_gs(path)
    return _gcs_client.bucket(b).blob(k).download_as_text()

def gcs_exists(path: str) -> bool:
    b, k = _split_gs(path)
    return _gcs_client.bucket(b).blob(k).exists()

log.info("Config loaded | mode=%s | platforms=%s", PIPELINE_MODE, PLATFORMS_TO_INCLUDE)
log.info("Inputs:  %s | %s", COMMENTS_LLM_PATH, COMMENTS_ML_PATH)
log.info("Outputs: %s | %s", TAXONOMY_GCS_PATH, RESULTS_GCS_PATH)

In [ ]:
# Cell 3 — Vertex AI initialization (was MISSING in persona_fix.ipynb)
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)  # ADC resolves creds

def get_model(model_name: str) -> GenerativeModel:
    return GenerativeModel(model_name)

def get_generation_config(max_tokens: int) -> GenerationConfig:
    return GenerationConfig(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_output_tokens=max_tokens,
    )

log.info("Vertex AI initialized | project=%s | location=%s", GCP_PROJECT_ID, GCP_LOCATION)

In [ ]:
# Cell 4 — Load comments (was MISSING: comments_df was undefined)
def load_comments(path: str, platforms: list) -> pd.DataFrame:
    log.info("Loading comments: %s", path)
    df = pd.read_json(path, lines=True)          # gcsfs handles gs://
    if "platform" not in df.columns:
        raise ValueError("comments_llm.jsonl has no 'platform' field")
    df["platform"] = df["platform"].astype(str).str.lower()
    keep = [p.lower() for p in platforms]
    before = len(df)
    df = df[df["platform"].isin(keep)].copy().reset_index(drop=True)
    log.info("Loaded %d comments (%d after platform filter)", before, len(df))
    return df

comments_df = load_comments(COMMENTS_LLM_PATH, PLATFORMS_TO_INCLUDE)
log.info("Platform breakdown:\n%s", comments_df["platform"].value_counts().to_string())

In [ ]:
# Cell 5 — Feature engineering: unified comments_ml.parquet, else compute from text
def load_ml_features(path: str, platforms: list) -> Optional[pd.DataFrame]:
    """Load the ONE unified ML parquet and filter by platform (has media_id)."""
    log.info("Attempting pre-computed ML features: %s", path)
    try:
        df = pd.read_parquet(path)               # gcsfs handles gs://
    except Exception as e:
        log.warning("ML parquet unavailable (%s) -> compute from text", e)
        return None
    if "platform" not in df.columns:
        log.warning("ML parquet missing 'platform' -> compute from text")
        return None
    df["platform"] = df["platform"].astype(str).str.lower()
    keep = [p.lower() for p in platforms]
    df = df[df["platform"].isin(keep)].copy().reset_index(drop=True)
    log.info("Loaded %d ML feature rows (media_id present: %s)",
             len(df), "media_id" in df.columns)
    return df

def compute_features_from_text(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Computing features from text for %d comments...", len(df))
    def feats(text):
        if not isinstance(text, str):
            return dict(text_length=0, word_count=0, emoji_count=0, unique_emoji_count=0,
                        mention_count=0, hashtag_count=0, url_count=0, has_links=0, has_numbers=0)
        return dict(
            text_length=len(text),
            word_count=len(text.split()),
            emoji_count=emoji_lib.emoji_count(text),
            unique_emoji_count=len({e["emoji"] for e in emoji_lib.emoji_list(text)}),
            mention_count=len(re.findall(r"@\w+", text)),
            hashtag_count=len(re.findall(r"#\w+", text)),
            url_count=len(re.findall(r"https?://\S+|www\.\S+", text)),
            has_links=int(bool(re.search(r"https?://\S+|www\.\S+", text))),
            has_numbers=int(bool(re.search(r"\d", text))),
        )
    base_cols = [c for c in ["comment_id", "author_id", "platform", "timestamp", "text"] if c in df.columns]
    features = df["text"].apply(feats).apply(pd.Series)
    return pd.concat([df[base_cols].reset_index(drop=True), features.reset_index(drop=True)], axis=1)

comment_features = load_ml_features(COMMENTS_ML_PATH, PLATFORMS_TO_INCLUDE)
if comment_features is None:
    comment_features = compute_features_from_text(comments_df)
elif "text" not in comment_features.columns:
    comment_features = comment_features.merge(
        comments_df[["comment_id", "text"]], on="comment_id", how="left")

log.info("Comment features ready: %d rows | cols=%s",
         len(comment_features), list(comment_features.columns)[:12])

In [ ]:
# Cell 6 — Aggregate to user level (single, correct definition; guarded metrics)
def build_user_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols = df.columns.tolist()
    df["user_platform_key"] = df["author_id"].astype(str) + "_" + df["platform"].astype(str)

    agg = {
        "author_id": ("author_id", "first"),
        "platform":  ("platform", "first"),
        "total_comments": ("comment_id", "count"),
    }
    if "media_id" in cols:
        agg["unique_posts_commented"] = ("media_id", "nunique")
        log.info("Using media_id for unique_posts_commented")
    else:
        agg["unique_posts_commented"] = ("comment_id", "count")
        log.warning("media_id absent -> unique_posts_commented = total_comments (conservative)")

    optional = {"text_length": "mean", "word_count": "mean", "emoji_count": "mean",
                "unique_emoji_count": "mean", "mention_count": "mean", "hashtag_count": "mean",
                "url_count": "mean", "has_links": "mean", "has_numbers": "mean"}
    for c, fn in optional.items():
        if c in cols:
            agg[f"mean_{c}"] = (c, fn)

    uf = df.groupby("user_platform_key").agg(**agg).reset_index()

    # Guarded derived metrics (no KeyError if a source column was absent)
    if {"mean_emoji_count", "mean_word_count"}.issubset(uf.columns):
        uf["emoji_usage_rate"] = uf["mean_emoji_count"] / uf["mean_word_count"].clip(lower=1)
    else:
        uf["emoji_usage_rate"] = 0.0
    uf["link_inclusion_rate"] = uf["mean_has_links"] * 100 if "mean_has_links" in uf.columns else 0.0
    uf["post_concentration_ratio"] = (
        uf["unique_posts_commented"] / uf["total_comments"].clip(lower=1)
    ).clip(upper=1.0)

    if "text" in cols:
        sort_col = "text_length" if "text_length" in cols else None
        ordered = df.sort_values(sort_col, ascending=False) if sort_col else df
        top = ordered.groupby("user_platform_key").head(5)
        top_agg = (top.groupby("user_platform_key")["text"]
                   .apply(lambda ts: " ||| ".join(str(t)[:200] for t in ts.tolist()))
                   .reset_index())
        top_agg.columns = ["user_platform_key", "top_comments_sample"]
        uf = uf.merge(top_agg, on="user_platform_key", how="left")

    return uf.drop(columns=["user_platform_key"])

user_features = build_user_feature_matrix(comment_features)
log.info("User feature matrix: %d users", len(user_features))
for p in sorted(user_features["platform"].unique()):
    n = int((user_features["platform"] == p).sum())
    log.info("  %-10s %d users (%.1f%%)", p, n, n / len(user_features) * 100)

In [ ]:
# Cell 7 — Stage 1: exploratory taxonomy discovery (Gemini Flash)
STAGE1_SYSTEM_PROMPT = """You are an expert community analyst for Show Reel Media Group.
Your task is to identify distinct audience persona archetypes from user comment behavior.

You will receive user profiles with behavioral metrics, platform, and sample comments.
Identify RECURRING behavioral patterns. Output a JSON array of persona candidates.
Schema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]
Output ONLY valid JSON. No preamble, no markdown fences."""

def _strip_fences(raw: str) -> str:
    raw = re.sub(r"^```json\s*", "", raw.strip())
    return re.sub(r"\s*```$", "", raw).strip()

def format_user_profile_for_stage1(row: pd.Series) -> str:
    return (f"USER: {str(row['author_id'])[:12]}... | {str(row.get('platform','?')).upper()} | "
            f"Comments: {int(row.get('total_comments', 0))} | "
            f"Emoji rate: {float(row.get('emoji_usage_rate', 0)):.0%} | "
            f"Sample: {str(row.get('top_comments_sample',''))[:300]}")

def run_stage1_taxonomy_discovery(user_features_df, n_sample=SAMPLE_N_USERS,
                                  batch_size=BATCH_SIZE_STAGE1, seed=SAMPLE_SEED) -> list:
    log.info("STAGE 1 | sample=%d | model=%s", n_sample, MODEL_STAGE1_EXPLORATORY)
    sample_df = user_features_df.sample(n=min(n_sample, len(user_features_df)),
                                        random_state=seed).reset_index(drop=True)
    model = get_model(MODEL_STAGE1_EXPLORATORY)
    config = get_generation_config(MAX_OUTPUT_TOKENS_STAGE1)
    batches = [sample_df.iloc[i:i+batch_size] for i in range(0, len(sample_df), batch_size)]

    candidates = []
    for bi, batch in enumerate(tqdm(batches, desc="Stage 1")):
        profiles = "\n\n".join(format_user_profile_for_stage1(r) for _, r in batch.iterrows())
        prompt = f"{STAGE1_SYSTEM_PROMPT}\n\n--- BATCH {bi+1}/{len(batches)} ---\n\n{profiles}\n\nOutput ONLY the JSON array."
        try:
            resp = model.generate_content(prompt, generation_config=config)
            parsed = json.loads(_strip_fences(resp.text))
            if isinstance(parsed, list):
                candidates.extend(parsed)
        except Exception as e:
            log.warning("Stage 1 batch %d failed: %s", bi + 1, str(e)[:120])
        time.sleep(1.5)
    log.info("STAGE 1 complete | %d candidates", len(candidates))
    return candidates

def save_taxonomy_for_review(candidates: list, path=TAXONOMY_GCS_PATH):
    payload = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": "Consolidate raw_candidates into a MECE final_taxonomy, then set status='APPROVED'.",
        "raw_candidates": candidates,
        "final_taxonomy": [],
    }
    gcs_write_text(path, json.dumps(payload, ensure_ascii=False, indent=2))
    log.info("Taxonomy candidates written to %s (HUMAN REVIEW REQUIRED)", path)

log.info("Stage 1 functions ready")

In [ ]:
# Cell 8 — Stage 2: deterministic classification (Gemini Pro)
def build_stage2_system_prompt(taxonomy: list) -> str:
    block = ""
    for p in taxonomy:
        block += (f"\nPERSONA: {p.get('codename','UNKNOWN')}\n"
                  f"  Description: {p.get('description','N/A')}\n"
                  f"  Signals: {'; '.join(p.get('signals', []))}\n")
    return (f"You are a community analyst. Classify each user into exactly ONE persona:\n{block}\n"
            "RULES:\n1. Exactly ONE persona per user.\n2. Output confidence 0.0-1.0.\n"
            "3. Cite evidence.\n4. Output ONLY a valid JSON array of objects: "
            '{"author_id": str, "platform": str, "persona_codename": str, '
            '"confidence": float, "justification": str}')

def format_user_profile_for_stage2(row: pd.Series) -> dict:
    return {
        "author_id": str(row["author_id"]),
        "platform": str(row.get("platform", "unknown")),
        "total_comments": int(row.get("total_comments", 0)),
        "emoji_usage_rate": round(float(row.get("emoji_usage_rate", 0)), 2),
        "link_inclusion_rate": round(float(row.get("link_inclusion_rate", 0)), 1),
        "sample_comments": str(row.get("top_comments_sample", ""))[:300],
    }

def run_stage2_classification(user_features_df, taxonomy, batch_size=BATCH_SIZE_STAGE2,
                              output_path=RESULTS_GCS_PATH) -> pd.DataFrame:
    log.info("STAGE 2 | users=%d | model=%s", len(user_features_df), MODEL_STAGE2_CLASSIFY)
    system_prompt = build_stage2_system_prompt(taxonomy)
    model = get_model(MODEL_STAGE2_CLASSIFY)
    config = get_generation_config(MAX_OUTPUT_TOKENS_STAGE2)
    batches = [user_features_df.iloc[i:i+batch_size] for i in range(0, len(user_features_df), batch_size)]

    results = []
    for bi, batch in enumerate(tqdm(batches, desc="Stage 2")):
        profiles = [format_user_profile_for_stage2(r) for _, r in batch.iterrows()]
        prompt = (f"{system_prompt}\n\nBATCH {bi+1}/{len(batches)}:\n"
                  f"{json.dumps(profiles, ensure_ascii=False)}\n\nClassify. Output ONLY a JSON array.")
        try:
            resp = model.generate_content(prompt, generation_config=config)
            parsed = json.loads(_strip_fences(resp.text))
            if isinstance(parsed, list):
                results.extend(parsed)
        except Exception as e:
            log.warning("Stage 2 batch %d failed: %s", bi + 1, str(e)[:120])
            for prof in profiles:
                results.append({"author_id": prof["author_id"], "platform": prof["platform"],
                                "persona_codename": "ERROR", "confidence": 0.0,
                                "justification": "API error"})
        # checkpoint to GCS so an interrupted scheduled run is recoverable
        if (bi + 1) % 10 == 0:
            try:
                pd.DataFrame(results).to_parquet(CHECKPOINT_GCS_PATH, index=False)
            except Exception as e:
                log.warning("Checkpoint write failed: %s", str(e)[:80])
        time.sleep(1.0)

    res = pd.DataFrame(results)
    # one row per (author_id, platform) before merge -> no row explosion
    if not res.empty:
        res = res.drop_duplicates(subset=["author_id", "platform"], keep="first")
    final_df = user_features_df.merge(
        res[["author_id", "platform", "persona_codename", "confidence"]],
        on=["author_id", "platform"], how="left")

    final_df.to_parquet(output_path, index=False)   # gcsfs handles gs://
    log.info("STAGE 2 complete | %d users -> %s", len(final_df), output_path)
    log.info("Distribution:\n%s", final_df["persona_codename"].value_counts().head(10).to_string())
    return final_df

log.info("Stage 2 functions ready")

In [ ]:
# Cell 9 — Orchestrator with fatal-error guard (re-raises so scheduler marks FAILED)
def load_approved_taxonomy(path=TAXONOMY_GCS_PATH) -> list:
    if not gcs_exists(path):
        raise FileNotFoundError(f"Taxonomy not found: {path}")
    data = json.loads(gcs_read_text(path))
    if data.get("status") != "APPROVED":
        raise ValueError(f"Taxonomy status is {data.get('status')!r}; requires 'APPROVED'.")
    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty.")
    log.info("Approved taxonomy loaded | %d personas", len(taxonomy))
    return taxonomy

def run_pipeline(mode: str):
    log.info("PIPELINE mode=%s | platforms=%s", mode, PLATFORMS_TO_INCLUDE)

    if mode in ("SAMPLE", "ALL"):
        candidates = run_stage1_taxonomy_discovery(user_features)
        save_taxonomy_for_review(candidates, TAXONOMY_GCS_PATH)
        if mode == "SAMPLE":
            log.info("STAGE 1 done — awaiting human review of %s "
                     "(set final_taxonomy + status=APPROVED, then run with PIPELINE_MODE=FULL)",
                     TAXONOMY_GCS_PATH)
            return None

    if mode in ("FULL", "ALL"):
        taxonomy = load_approved_taxonomy(TAXONOMY_GCS_PATH)
        return run_stage2_classification(user_features, taxonomy, output_path=RESULTS_GCS_PATH)

    return None

def main():
    log.info("=== PERSONA PIPELINE START | mode=%s ===", PIPELINE_MODE)
    run_pipeline(PIPELINE_MODE)
    log.info("=== PERSONA PIPELINE END ===")

try:
    main()
except Exception:
    log.error("FATAL — pipeline aborted:\n%s", traceback.format_exc())
    raise   # re-raise so Colab Enterprise marks the scheduled run FAILED

In [ ]:
# Cell 10 — Validation / QA (reads results from GCS)
def validate_results(path=RESULTS_GCS_PATH):
    if not gcs_exists(path):
        log.warning("Results not found: %s (run Stage 2 first)", path)
        return
    df = pd.read_parquet(path)
    total = len(df)
    ok = df["persona_codename"].notna() & (df["persona_codename"] != "ERROR")
    log.info("QA | total=%d | classified=%d (%.1f%%) | errors=%d",
             total, int(ok.sum()), ok.sum() / total * 100, int((~ok).sum()))
    if "confidence" in df.columns:
        conf = pd.to_numeric(df.loc[ok, "confidence"], errors="coerce")
        log.info("Confidence | mean=%.3f | median=%.3f", conf.mean(), conf.median())
    log.info("Persona distribution:\n%s", df["persona_codename"].value_counts().head(10).to_string())

validate_results()